# microWakeWord Custom Trainer — Colab Edition

Train a custom wake word model for [ESPHome](https://esphome.io) and [Home Assistant](https://www.home-assistant.io/) using [microWakeWord](https://github.com/kahrendt/microWakeWord).

**Run each cell in order. Two required restarts — the notebook tells you when.**

Optionally provide your own voice recordings for better accuracy (Step 7b).

## Step 0: Free Disk Space
Colab has limited disk. Remove unused system files first.

In [ ]:
import shutil, os, glob
for path in ['/usr/share/doc', '/usr/share/man', '/usr/share/locale',
             '/usr/lib/google-cloud-sdk', '/usr/local/android-sdk',
             '/content/sample_data']:
    shutil.rmtree(path, ignore_errors=True)
for p in glob.glob('/usr/local/julia-*'):
    shutil.rmtree(p, ignore_errors=True)
!apt-get clean -qq && apt-get autoremove -y -qq
!pip cache purge -q 2>/dev/null
print('Disk after cleanup:')
!df -h / | tail -1

## Step 1: Check GPU
Make sure the runtime is set to **T4 GPU** (Runtime → Change runtime type).

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout or 'nvidia-smi not found — make sure T4 GPU runtime is selected')

## Step 2a: Install All Packages

Run this cell, then **restart once** (Runtime → Restart session). Continue from **Step 2b**.

In [ ]:
print('Installing system packages...')
!apt-get install -y -q espeak-ng libespeak-ng-dev ffmpeg

print('\nInstalling piper-tts...')
!pip install piper-tts
!python -c "from piper import PiperVoice, SynthesisConfig; print('piper: OK')"

print('\nRemoving conflicting pre-installs...')
!pip uninstall -y jax jaxlib tensorstore tensorflow-decision-forests tensorflow-text \
    opencv-python opencv-python-headless opencv-contrib-python shap ydf grain \
    pytensor xarray-einstats rasterio tobler cupy-cuda12x 2>/dev/null

print('\nInstalling TensorFlow 2.18 (CUDA 12.8 compatible)...')
!pip install --quiet --timeout=300 'tensorflow[and-cuda]==2.18.0' protobuf==4.25.3 ml-dtypes==0.3.2

print('\nInstalling remaining deps...')
!pip install --quiet onnxruntime pyyaml datasets mmap-ninja tqdm \
    audiomentations webrtcvad-wheels huggingface_hub soundfile librosa

print('\n✅ Done! Restart: Runtime → Restart session, then continue from Step 2b.')

## Step 2b: Pin numpy & scipy

A **"Restart required"** popup will appear — **click it immediately**. Then continue from **Step 3**.

In [ ]:
!pip uninstall -y numpy scipy 2>/dev/null
!find /usr/local/lib/python3.*/dist-packages -name '*.pyc' -path '*/numpy/*' -delete 2>/dev/null
!find /usr/local/lib/python3.*/dist-packages -name '*.pyc' -path '*/scipy/*' -delete 2>/dev/null
!pip install --force-reinstall --no-cache-dir numpy==1.26.4 scipy==1.13.1
print('\n✅ numpy + scipy pinned. Click the Restart button now, then continue from Step 3.')

## ⚠️ Restart Required

**Runtime → Restart session** (Ctrl+M .) → continue from **Step 3**. Both restarts are now done.

## Step 3: Verify Installs & Clone Repositories

In [ ]:
import subprocess, os, sys, shutil

print(f'Python: {sys.version}')

import numpy as np
assert np.__version__ == '1.26.4', f'Bad numpy: {np.__version__}'
print(f'numpy {np.__version__}: OK')

import scipy; print(f'scipy {scipy.__version__}: OK')
import tensorflow as tf; print(f'tensorflow {tf.__version__}: OK')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU visible to TF: {gpus[0].name} ✅')
else:
    print('❌ TF cannot see GPU — check runtime type is T4 GPU')

r = subprocess.run([sys.executable, '-c',
    'from piper import PiperVoice, SynthesisConfig; print("piper: OK")'],
    capture_output=True, text=True)
print(r.stdout.strip() if r.returncode == 0 else f'piper FAILED:\n{r.stderr}')

# Always re-clone fresh
if os.path.exists('microWakeWord'):
    shutil.rmtree('microWakeWord')
subprocess.run(['git', 'clone',
    'https://github.com/kahrendt/microWakeWord.git'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
    'microWakeWord'], check=True)

# ── Patch train.py for Keras 3 compatibility ──────────────────
#
# Two problems with the original validate_nonstreaming() on Keras 3 / TF 2.18:
#
#   1. Hardcoded batch_size=1024 in model.evaluate() conflicts with the
#      model's Input layer static batch_size (e.g. 128). Keras 3 enforces this.
#
#   2. Passing raw NumPy arrays to model.evaluate() triggers Keras 3's
#      internal array→Dataset conversion which mishandles shapes.
#
# Fix: Replace validate_nonstreaming entirely.
#   - Use tf.data.Dataset.from_tensor_slices().batch() for explicit batching
#   - Use drop_remainder=True so every batch matches the Input layer's static shape
#   - Handle Keras 3 returning dict vs list from model.evaluate()

CUT_MARKER = '\n# === NOTEBOOK PATCH ==='
train_py = 'microWakeWord/microwakeword/train.py'
with open(train_py, 'r') as f:
    src = f.read()
if CUT_MARKER in src:
    src = src[:src.index(CUT_MARKER)]

patch = CUT_MARKER + r'''
import tensorflow as _tf_patch

def validate_nonstreaming(config, data_processor, model, test_set):
    """Keras 3-compatible replacement for validate_nonstreaming."""
    testing_fingerprints, testing_ground_truth, _ = data_processor.get_data(
        test_set,
        batch_size=config["batch_size"],
        features_length=config["spectrogram_length"],
        truncation_strategy="truncate_start",
    )
    testing_ground_truth = testing_ground_truth.reshape(-1, 1)

    n_samples = testing_fingerprints.shape[0] if hasattr(testing_fingerprints, 'shape') else len(testing_fingerprints)
    if n_samples == 0:
        return {
            "accuracy": 0, "recall": 0, "precision": 0, "auc": 0, "loss": 99,
            "recall_at_no_faph": 0, "cutoff_for_no_faph": 0,
            "ambient_false_positives": 0, "ambient_false_positives_per_hour": 0,
            "average_viable_recall": 0,
        }

    model.reset_metrics()

    bs = config["batch_size"]
    ds = _tf_patch.data.Dataset.from_tensor_slices(
        (testing_fingerprints, testing_ground_truth)
    ).batch(bs, drop_remainder=True)

    result = model.evaluate(ds, return_dict=True, verbose=0)

    if isinstance(result, (list, tuple)):
        names = [m.name for m in model.metrics]
        result = dict(zip(names, result))

    def _as_numpy(v):
        return v.numpy() if hasattr(v, 'numpy') else np.array(v)

    metrics = {}
    metrics["accuracy"] = result["accuracy"]
    metrics["recall"] = result["recall"]
    metrics["precision"] = result["precision"]
    metrics["auc"] = result["auc"]
    metrics["loss"] = result["loss"]
    metrics["recall_at_no_faph"] = 0
    metrics["cutoff_for_no_faph"] = 0
    metrics["ambient_false_positives"] = 0
    metrics["ambient_false_positives_per_hour"] = 0
    metrics["average_viable_recall"] = 0

    test_set_fp = _as_numpy(result["fp"])

    if data_processor.get_mode_size("validation_ambient") > 0:
        (
            ambient_testing_fingerprints,
            ambient_testing_ground_truth,
            _,
        ) = data_processor.get_data(
            test_set + "_ambient",
            batch_size=config["batch_size"],
            features_length=config["spectrogram_length"],
            truncation_strategy="split",
        )
        ambient_testing_ground_truth = ambient_testing_ground_truth.reshape(-1, 1)

        amb_ds = _tf_patch.data.Dataset.from_tensor_slices(
            (ambient_testing_fingerprints, ambient_testing_ground_truth)
        ).batch(bs, drop_remainder=True)

        with swap_attribute(model, "reset_metrics", lambda: None):
            ambient_predictions = model.evaluate(
                amb_ds, return_dict=True, verbose=0,
            )

        if isinstance(ambient_predictions, (list, tuple)):
            names = [m.name for m in model.metrics]
            ambient_predictions = dict(zip(names, ambient_predictions))

        duration_of_ambient_set = (
            data_processor.get_mode_duration("validation_ambient") / 3600.0
        )

        all_true_positives = _as_numpy(ambient_predictions["tp"])
        ambient_false_positives = _as_numpy(ambient_predictions["fp"]) - test_set_fp
        all_false_negatives = _as_numpy(ambient_predictions["fn"])

        metrics["auc"] = ambient_predictions["auc"]
        metrics["loss"] = ambient_predictions["loss"]

        recall_at_cutoffs = (
            all_true_positives / (all_true_positives + all_false_negatives)
        )
        faph_at_cutoffs = ambient_false_positives / duration_of_ambient_set

        target_faph_cutoff_probability = 1.0
        recall_at_no_faph = 0
        for index, cutoff in enumerate(np.linspace(0.0, 1.0, 101)):
            if faph_at_cutoffs[index] == 0:
                target_faph_cutoff_probability = cutoff
                recall_at_no_faph = recall_at_cutoffs[index]
                break

        if faph_at_cutoffs[0] > 2:
            index_of_first_viable = 1
            while faph_at_cutoffs[index_of_first_viable] > 2:
                index_of_first_viable += 1
            x0 = faph_at_cutoffs[index_of_first_viable - 1]
            y0 = recall_at_cutoffs[index_of_first_viable - 1]
            x1 = faph_at_cutoffs[index_of_first_viable]
            y1 = recall_at_cutoffs[index_of_first_viable]
            recall_at_2faph = (y0 * (x1 - 2.0) + y1 * (2.0 - x0)) / (x1 - x0)
        else:
            index_of_first_viable = 0
            recall_at_2faph = recall_at_cutoffs[0]

        x_coordinates = [2.0]
        y_coordinates = [recall_at_2faph]
        for index in range(index_of_first_viable, len(recall_at_cutoffs)):
            if faph_at_cutoffs[index] != x_coordinates[-1]:
                x_coordinates.append(faph_at_cutoffs[index])
                y_coordinates.append(recall_at_cutoffs[index])

        average_viable_recall = (
            np.trapz(np.flip(y_coordinates), np.flip(x_coordinates)) / 2.0
        )

        metrics["recall_at_no_faph"] = recall_at_no_faph
        metrics["cutoff_for_no_faph"] = target_faph_cutoff_probability
        metrics["ambient_false_positives"] = ambient_false_positives[50]
        metrics["ambient_false_positives_per_hour"] = faph_at_cutoffs[50]
        metrics["average_viable_recall"] = average_viable_recall

    return metrics
'''

with open(train_py, 'w') as f:
    f.write(src + patch)
print('microWakeWord cloned + Keras 3 patched ✅')

if not os.path.exists('piper-sample-generator'):
    subprocess.run(['git', 'clone',
        'https://github.com/rhasspy/piper-sample-generator.git'], check=True)
print('piper-sample-generator: OK')
print('\n✅ All verified and ready!')

## Step 4: Download Piper Voice Model

In [ ]:
import os, urllib.request
os.makedirs('piper-sample-generator/models', exist_ok=True)
url  = ('https://github.com/rhasspy/piper-sample-generator/releases/'
        'download/v2.0.0/en_US-libritts_r-medium.pt')
model_path = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'
if not os.path.exists(model_path):
    print('Downloading Piper model...')
    urllib.request.urlretrieve(url, model_path)
    print('✅ Done!')
else:
    print('✅ Already exists')

## Step 5: Configure

Edit `TARGET_WORD` below and re-run Step 6 until the pronunciation sounds right.

**Tips:** underscores between syllables (`hey_air_uh_gorn`), `sh`/`ch`/`th` for those sounds, `ee`/`oo` for long vowels.

**Hugging Face Token (optional):** The negative datasets (~3 GB) are downloaded from Hugging Face. Without a token, downloads are rate-limited and may be slow. Adding a free token speeds things up significantly.

1. Create a free account at [huggingface.co](https://huggingface.co/join)
2. Go to [Settings → Access Tokens](https://huggingface.co/settings/tokens)
3. Click **Create new token** → give it a name → set type to **Read** → click **Generate**
4. Copy the token (starts with `hf_...`) and paste it into `HF_TOKEN` below

**GitHub (optional):** Set `GITHUB_TOKEN` and `GITHUB_REPO` to push the trained model to your repo automatically. Leave empty to just download locally.

In [ ]:
# ── Required ──────────────────────────────────────────────────
TARGET_WORD    = 'hey_air_uh_gorn'
NUM_SAMPLES    = 1000
TRAINING_STEPS = 10000

# ── Optional: Hugging Face token (faster downloads) ──────────
# Get a free token at https://huggingface.co/settings/tokens
# Set type to "Read". Paste the hf_... token below.
HF_TOKEN = ''   # e.g. 'hf_xxxxxxxxxxxx'

# ── Optional: GitHub push ─────────────────────────────────────
# Leave empty to skip GitHub push and just download the model.
GITHUB_TOKEN = ''   # e.g. 'ghp_xxxxxxxxxxxx'
GITHUB_REPO  = ''   # e.g. 'YourUser/your-repo-name'

print(f'Word:    {TARGET_WORD}')
print(f'Samples: {NUM_SAMPLES}')
print(f'Steps:   {TRAINING_STEPS}')
if HF_TOKEN:
    print(f'HF:      token set (faster downloads)')
else:
    print('HF:      no token (downloads may be rate-limited)')
if GITHUB_TOKEN and GITHUB_REPO:
    print(f'GitHub:  {GITHUB_REPO} (push enabled)')
else:
    print('GitHub:  not configured (model will be downloaded locally)')

## Step 6: Generate Test Sample
Listen to the audio below. If it sounds wrong, change `TARGET_WORD` in Step 5 and re-run this cell.

In [ ]:
import subprocess, os, sys, shutil
from IPython.display import Audio, display

preview_dir = '/tmp/preview_sample'
if os.path.exists(preview_dir):
    shutil.rmtree(preview_dir)
os.makedirs(preview_dir)

env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD, '--max-samples', '1', '--batch-size', '1',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', preview_dir]
result = subprocess.run(cmd, capture_output=True, text=True, env=env)
if result.returncode == 0:
    wavs = sorted([f for f in os.listdir(preview_dir) if f.endswith('.wav')])
    if wavs:
        print(f'Phonetic spelling: "{TARGET_WORD}"')
        print('Adjust TARGET_WORD in Step 5 if this sounds wrong.\n')
        display(Audio(os.path.join(preview_dir, wavs[0])))
else:
    print('STDOUT:', result.stdout)
    print('STDERR:', result.stderr)

## Step 7: Generate All Synthetic Samples

In [ ]:
import subprocess, os, sys, shutil

# Check for real recordings to determine split
REAL_DIR = 'real_recordings'
os.makedirs(REAL_DIR, exist_ok=True)
real_audio_files = [
    os.path.join(REAL_DIR, f) for f in os.listdir(REAL_DIR)
    if f.lower().endswith(('.wav', '.mp3', '.m4a', '.flac'))
]
USING_REAL = len(real_audio_files) > 0
SYNTHETIC_COUNT = NUM_SAMPLES // 2 if USING_REAL else NUM_SAMPLES
REAL_TARGET     = NUM_SAMPLES // 2 if USING_REAL else 0

if USING_REAL:
    print(f'Found {len(real_audio_files)} real recording(s) — using 50/50 split')
    print(f'  Synthetic: {SYNTHETIC_COUNT} | Real augmented: {REAL_TARGET}')
else:
    print(f'No real recordings found — using 100% synthetic ({SYNTHETIC_COUNT} samples)')

# Clean previous samples
if os.path.exists('generated_samples'):
    shutil.rmtree('generated_samples')
os.makedirs('generated_samples')

print(f'\nGenerating {SYNTHETIC_COUNT} synthetic samples...')
env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD, '--max-samples', str(SYNTHETIC_COUNT), '--batch-size', '100',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', 'generated_samples']
result = subprocess.run(cmd, capture_output=True, text=True, env=env)
if result.returncode == 0:
    synth_count = len([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    print(f'✅ Generated {synth_count} synthetic samples!')
else:
    print('STDOUT:', result.stdout)
    print('STDERR:', result.stderr)

## Step 7b: Real Recordings (Optional)

Skip this step if you don't have your own voice recordings.

**Option A — Upload from your computer:** Run the next cell and use the upload widget.

**Option B — Google Drive:** Put recordings in a Drive folder, mount, and copy.

Supported formats: `.wav`, `.mp3`, `.m4a`, `.flac`

**Recording tips:** One long file with 10-50 repetitions, 1-2 second pause between each. Phone or laptop mic is fine.

In [ ]:
import os, shutil, glob

REAL_DIR = 'real_recordings'
os.makedirs(REAL_DIR, exist_ok=True)

# ── Choose ONE method below (comment out the other) ────────────

# --- Option A: Upload widget ---
from google.colab import files
print('Select your recording file(s):')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = os.path.join(REAL_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved: {dest} ({len(data)/1024:.1f} KB)')

# --- Option B: Google Drive (uncomment below, comment out Option A) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_FOLDER = '/content/drive/MyDrive/wake_word_recordings'  # ← edit this path
# for f in glob.glob(os.path.join(DRIVE_FOLDER, '*')):
#     if f.lower().endswith(('.wav', '.mp3', '.m4a', '.flac')):
#         shutil.copy(f, REAL_DIR)
#         print(f'  Copied: {os.path.basename(f)}')

# ── Summary ───────────────────────────────────────────────────
real_files = [f for f in os.listdir(REAL_DIR)
              if f.lower().endswith(('.wav', '.mp3', '.m4a', '.flac'))]
print(f'\n✅ {len(real_files)} recording file(s) in {REAL_DIR}/')

In [ ]:
import os, sys, numpy as np
import soundfile as sf
import librosa

REAL_DIR = 'real_recordings'
real_audio_files = [
    os.path.join(REAL_DIR, f) for f in os.listdir(REAL_DIR)
    if f.lower().endswith(('.wav', '.mp3', '.m4a', '.flac'))
]

if not real_audio_files:
    print('No real recordings found — skipping. (This is fine, synthetic-only works.)')
else:
    # Recalculate split now that we have real recordings
    USING_REAL = True
    REAL_TARGET = NUM_SAMPLES // 2

    def adaptive_split(audio_np, sr, min_clip_ms=250, max_clip_ms=3500,
                        min_silence_ms=400, keep_silence_ms=80):
        """Split a recording into individual utterances using adaptive thresholding."""
        frame_ms  = 20
        frame_len = int(sr * frame_ms / 1000)
        if len(audio_np) < frame_len * 2:
            return []

        n_frames = len(audio_np) // frame_len
        frames = audio_np[:n_frames * frame_len].reshape(n_frames, frame_len)
        rms = np.sqrt(np.mean(frames ** 2, axis=1))

        rms_safe = np.clip(rms, 1e-10, None)
        rms_db   = 20 * np.log10(rms_safe)
        p10, p90 = np.percentile(rms_db, [10, 90])
        threshold_db = p10 + 0.35 * (p90 - p10)
        threshold    = 10 ** (threshold_db / 20)
        print(f'    Adaptive threshold: {threshold_db:.1f} dB  '
              f'(noise floor ~{p10:.1f} dB, speech peak ~{p90:.1f} dB)')

        is_speech = rms > threshold
        min_sil_frames  = max(1, min_silence_ms // frame_ms)
        keep_frames     = max(1, keep_silence_ms // frame_ms)
        min_clip_frames = max(1, min_clip_ms // frame_ms)
        max_clip_frames = max(1, max_clip_ms // frame_ms)

        clips = []
        i = 0
        while i < n_frames:
            if is_speech[i]:
                start = max(0, i - keep_frames)
                j = i
                while j < n_frames:
                    if not is_speech[j]:
                        sil_start = j
                        while j < n_frames and not is_speech[j]:
                            j += 1
                        if j - sil_start >= min_sil_frames:
                            end = min(n_frames, sil_start + keep_frames)
                            clips.append((start, end))
                            i = j
                            break
                    else:
                        j += 1
                else:
                    clips.append((start, min(n_frames, j)))
                    i = j
            else:
                i += 1

        result = []
        too_short = too_long = 0
        for (s, e) in clips:
            dur_frames = e - s
            if dur_frames < min_clip_frames:
                too_short += 1; continue
            if dur_frames > max_clip_frames:
                too_long += 1; continue
            result.append(audio_np[s * frame_len : e * frame_len])

        if too_short or too_long:
            print(f'    Filtered: {too_short} too short (<{min_clip_ms}ms), '
                  f'{too_long} too long (>{max_clip_ms}ms)')
        return result

    # Extract clips from all recording files
    raw_clips = []
    for fpath in real_audio_files:
        print(f'  Loading {os.path.basename(fpath)}...')
        audio_np, sr = librosa.load(fpath, sr=16000, mono=True)
        print(f'    Duration: {len(audio_np)/sr:.1f}s')
        clips = adaptive_split(audio_np, sr)
        print(f'    {len(clips)} valid clips extracted')
        raw_clips.extend(clips)

    print(f'  Total clips: {len(raw_clips)}')

    if len(raw_clips) == 0:
        print('  No valid clips detected — continuing with synthetic-only.')
        USING_REAL = False
    else:
        reps_per_clip = max(1, -(-REAL_TARGET // len(raw_clips)))
        if reps_per_clip > 200:
            print(f'  WARNING: Only {len(raw_clips)} clip(s) — capping augmentation at 200x.')
            reps_per_clip = 200
            REAL_TARGET = min(REAL_TARGET, len(raw_clips) * (reps_per_clip + 1))
            print(f'  Adjusted real target: {REAL_TARGET}')

        augmented_count = 0
        for i, clip in enumerate(raw_clips):
            if augmented_count >= REAL_TARGET:
                break
            orig_path = f'generated_samples/real_{i:04d}_orig.wav'
            sf.write(orig_path, clip, 16000)
            augmented_count += 1
            for rep in range(reps_per_clip):
                if augmented_count >= REAL_TARGET:
                    break
                aug = clip.copy()
                speed = np.random.uniform(0.85, 1.15)
                new_len = int(len(aug) / speed)
                aug = np.interp(np.linspace(0, len(aug)-1, new_len),
                                np.arange(len(aug)), aug).astype(np.float32)
                aug = aug * np.random.uniform(0.6, 1.4)
                aug = aug + (np.random.randn(len(aug)).astype(np.float32)
                             * np.random.uniform(0.0, 0.008))
                aug = np.clip(aug, -1.0, 1.0)
                sf.write(f'generated_samples/real_{i:04d}_aug{rep:03d}.wav',
                         aug, 16000)
                augmented_count += 1

        total_samples = len([f for f in os.listdir('generated_samples')
                             if f.endswith('.wav')])
        print(f'✅ {augmented_count} real/augmented clips written')
        print(f'✅ {total_samples} total samples in generated_samples/')

## Step 8: Download Augmentation Data
Room impulse responses + background noises (~1.3 GB).

In [ ]:
import os
os.makedirs('mit_rirs', exist_ok=True)
if not os.listdir('mit_rirs'):
    print('Downloading RIRs + background noises (~1.3 GB)...')
    !wget --progress=bar:force -O /tmp/rirs_noises.zip https://www.openslr.org/resources/28/rirs_noises.zip
    !unzip -q /tmp/rirs_noises.zip -d mit_rirs
    print('Extracted!')
else:
    print('Already downloaded')
noise_dir = 'mit_rirs/RIRS_NOISES/pointsource_noises'
if os.path.exists(noise_dir):
    n = len([f for f in os.listdir(noise_dir) if f.endswith('.wav')])
    print(f'✅ {n} background noise files ready')
else:
    print('❌ pointsource_noises not found')

### Disk Cleanup (After Step 8)

In [ ]:
import os, subprocess, sys
if os.path.exists('/tmp/rirs_noises.zip'):
    os.remove('/tmp/rirs_noises.zip'); print('Deleted rirs_noises.zip')
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], capture_output=True)
print('pip cache purged')
!df -h / | tail -1

## Step 9: Generate Spectrograms

In [ ]:
import os, sys
if 'microWakeWord' not in sys.path:
    sys.path.insert(0, 'microWakeWord')

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

MMAP_TRAIN = 'generated_augmented_features/training/wakeword_mmap'
MMAP_TEST  = 'generated_augmented_features/testing/wakeword_mmap'
os.makedirs(os.path.dirname(MMAP_TRAIN), exist_ok=True)
os.makedirs(os.path.dirname(MMAP_TEST),  exist_ok=True)

clips = Clips(input_directory='generated_samples', file_pattern='*.wav',
    max_clip_duration_s=None, remove_silence=False,
    random_split_seed=10, split_count=0.1)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1, 'TanhDistortion': 0.1,
        'PitchShift': 0.1, 'BandStopFilter': 0.1,
        'AddColorNoise': 0.1, 'AddBackgroundNoise': 0.75,
        'Gain': 1.0, 'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['mit_rirs/RIRS_NOISES/pointsource_noises'],
    background_min_snr_db=-5, background_max_snr_db=10,
    min_jitter_s=0.195, max_jitter_s=0.205,
)

spectrograms = SpectrogramGeneration(
    clips=clips, augmenter=augmenter, slide_frames=10, step_ms=10)

print('Writing train split...')
RaggedMmap.from_generator(
    out_dir=MMAP_TRAIN,
    sample_generator=spectrograms.spectrogram_generator(split='train', repeat=2),
    batch_size=100, verbose=True)

print('Writing test split...')
RaggedMmap.from_generator(
    out_dir=MMAP_TEST,
    sample_generator=spectrograms.spectrogram_generator(split='test', repeat=1),
    batch_size=100, verbose=True)

mmap_tr = RaggedMmap(MMAP_TRAIN)
mmap_te = RaggedMmap(MMAP_TEST)
assert len(mmap_tr) > 0, 'train mmap is empty — check generated_samples/ has .wav files.'
assert len(mmap_te) > 0, 'test mmap is empty — split_count=0.1 yielded 0 test clips; increase NUM_SAMPLES.'
print(f'✅ {len(mmap_tr)} train spectrograms | {len(mmap_te)} test spectrograms')
print(f'   Shape of first: {mmap_tr[0].shape}')

## Step 10: Download Negative Datasets

In [ ]:
import os, zipfile
from huggingface_hub import hf_hub_download, list_repo_files
os.makedirs('negative_datasets', exist_ok=True)
repo_id, repo_type = 'kahrendt/microwakeword', 'dataset'
hf_token = HF_TOKEN if HF_TOKEN else None
if hf_token:
    print('Using Hugging Face token for authenticated downloads')
else:
    print('No HF token — downloads may be slower (rate-limited)')
zip_files = [f for f in list_repo_files(repo_id, repo_type=repo_type, token=hf_token) if f.endswith('.zip')]
print(f'Found: {zip_files}')
for fname in zip_files:
    base = os.path.splitext(os.path.basename(fname))[0]
    if not os.path.exists(f'negative_datasets/{base}'):
        print(f'Downloading {fname}...')
        local = hf_hub_download(repo_id=repo_id, filename=fname, repo_type=repo_type, token=hf_token)
        with zipfile.ZipFile(local, 'r') as zf:
            zf.extractall('negative_datasets')
        print(f'  done: {base}')
    else:
        print(f'  exists: {base}')
neg_dirs = [d for d in os.listdir('negative_datasets')
            if os.path.isdir(f'negative_datasets/{d}')]
print(f'\n✅ Directories: {neg_dirs}')

### Disk Cleanup (After Step 10)

In [ ]:
import os, shutil
hf_cache = os.path.expanduser('~/.cache/huggingface')
if os.path.exists(hf_cache):
    size = sum(os.path.getsize(os.path.join(dp,f))
               for dp,_,fs in os.walk(hf_cache) for f in fs)
    shutil.rmtree(hf_cache)
    print(f'Deleted HuggingFace cache ({size/1e9:.1f} GB freed)')
else:
    print('No HuggingFace cache found')
!df -h / | tail -1

## Step 11: Create Training Config

In [ ]:
import yaml, os

neg_dirs   = sorted([d for d in os.listdir('negative_datasets')
                     if os.path.isdir(f'negative_datasets/{d}')
                     and not d.startswith('__')])
eval_dirs  = [d for d in neg_dirs if 'eval' in d]
train_dirs = [d for d in neg_dirs if 'eval' not in d]
print(f'Train negatives: {train_dirs}')
print(f'Eval negatives:  {eval_dirs}')

neg_features = []
for d in train_dirs:
    neg_features.append({'features_dir': f'negative_datasets/{d}',
        'sampling_weight': 10.0, 'penalty_weight': 1.0,
        'truth': False, 'truncation_strategy': 'random', 'type': 'mmap'})
for d in eval_dirs:
    neg_features.append({'features_dir': f'negative_datasets/{d}',
        'sampling_weight': 0.0, 'penalty_weight': 1.0,
        'truth': False, 'truncation_strategy': 'split', 'type': 'mmap'})

config = {
    'window_step_ms': 10,
    'train_dir': 'trained_models/wakeword',
    'spectrogram_length': 204,
    'stride': 3,
    'features': [
        {'features_dir': 'generated_augmented_features',
         'sampling_weight': 2.0, 'penalty_weight': 1.0,
         'truth': True, 'truncation_strategy': 'truncate_start', 'type': 'mmap'},
        {'features_dir': 'generated_augmented_features',
         'sampling_weight': 0.0, 'penalty_weight': 1.0,
         'truth': True, 'truncation_strategy': 'split', 'type': 'mmap'},
    ] + neg_features,
    'training_steps': [TRAINING_STEPS],
    'positive_class_weight': [1],
    'negative_class_weight': [20],
    'learning_rates': [0.001],
    'batch_size': 128,
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.9,
    'minimization_metric': '',
    'maximization_metric': 'average_viable_recall',
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print(f'✅ Config saved (batch_size={config["batch_size"]})')

## Step 12: Train Model

This takes ~1 hour per 10,000 steps on a T4 GPU. Progress prints every 500 steps.

In [ ]:
import sys, os, shutil, argparse, logging
import numpy as np

# Confirm GPU is visible
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'GPU: {gpus[0].name}')
else:
    raise RuntimeError('No GPU detected — go to Runtime → Change runtime type → T4 GPU, then re-run from Step 3.')

if os.path.exists('trained_models/wakeword'):
    shutil.rmtree('trained_models/wakeword')
    print('Cleared previous trained_models/wakeword')

# Import the microWakeWord training modules directly.
# The module uses argparse (not absl), and the training logic lives in
# the if __name__ == "__main__" block. We replicate that logic here
# so training runs in-process with full GPU access.
from microwakeword import model_train_eval as mte
from microwakeword import train as _train_mod
from microwakeword import mixednet, data as input_data
from absl import logging as absl_logging

absl_logging.set_verbosity(absl_logging.INFO)
absl_logging.use_python_logging()

# ── Monkey-patch validate_nonstreaming in the live module ─────
# The file-level patch from Step 3 may not be active if Python cached an
# earlier import. This ensures the Keras 3-compatible version is always
# used, regardless of import order or re-runs.

def _patched_validate(config, data_processor, model, test_set):
    """Keras 3-compatible validate_nonstreaming.
    Uses tf.data.Dataset instead of raw arrays to avoid Keras 3 shape bugs.
    Uses drop_remainder=True to match the model's static batch_size."""
    testing_fingerprints, testing_ground_truth, _ = data_processor.get_data(
        test_set,
        batch_size=config["batch_size"],
        features_length=config["spectrogram_length"],
        truncation_strategy="truncate_start",
    )
    testing_ground_truth = testing_ground_truth.reshape(-1, 1)

    n_samples = testing_fingerprints.shape[0] if hasattr(testing_fingerprints, 'shape') else len(testing_fingerprints)
    if n_samples == 0:
        return {
            "accuracy": 0, "recall": 0, "precision": 0, "auc": 0, "loss": 99,
            "recall_at_no_faph": 0, "cutoff_for_no_faph": 0,
            "ambient_false_positives": 0, "ambient_false_positives_per_hour": 0,
            "average_viable_recall": 0,
        }

    model.reset_metrics()

    bs = config["batch_size"]
    ds = tf.data.Dataset.from_tensor_slices(
        (testing_fingerprints, testing_ground_truth)
    ).batch(bs, drop_remainder=True)

    result = model.evaluate(ds, return_dict=True, verbose=0)

    if isinstance(result, (list, tuple)):
        names = [m.name for m in model.metrics]
        result = dict(zip(names, result))

    def _as_numpy(v):
        return v.numpy() if hasattr(v, 'numpy') else np.array(v)

    metrics = {}
    metrics["accuracy"] = result["accuracy"]
    metrics["recall"] = result["recall"]
    metrics["precision"] = result["precision"]
    metrics["auc"] = result["auc"]
    metrics["loss"] = result["loss"]
    metrics["recall_at_no_faph"] = 0
    metrics["cutoff_for_no_faph"] = 0
    metrics["ambient_false_positives"] = 0
    metrics["ambient_false_positives_per_hour"] = 0
    metrics["average_viable_recall"] = 0

    test_set_fp = _as_numpy(result["fp"])

    if data_processor.get_mode_size("validation_ambient") > 0:
        (
            ambient_testing_fingerprints,
            ambient_testing_ground_truth,
            _,
        ) = data_processor.get_data(
            test_set + "_ambient",
            batch_size=config["batch_size"],
            features_length=config["spectrogram_length"],
            truncation_strategy="split",
        )
        ambient_testing_ground_truth = ambient_testing_ground_truth.reshape(-1, 1)

        amb_ds = tf.data.Dataset.from_tensor_slices(
            (ambient_testing_fingerprints, ambient_testing_ground_truth)
        ).batch(bs, drop_remainder=True)

        _orig_reset = model.reset_metrics
        model.reset_metrics = lambda: None
        try:
            ambient_predictions = model.evaluate(
                amb_ds, return_dict=True, verbose=0,
            )
        finally:
            model.reset_metrics = _orig_reset

        if isinstance(ambient_predictions, (list, tuple)):
            names = [m.name for m in model.metrics]
            ambient_predictions = dict(zip(names, ambient_predictions))

        duration_of_ambient_set = (
            data_processor.get_mode_duration("validation_ambient") / 3600.0
        )

        all_true_positives = _as_numpy(ambient_predictions["tp"])
        ambient_false_positives = _as_numpy(ambient_predictions["fp"]) - test_set_fp
        all_false_negatives = _as_numpy(ambient_predictions["fn"])

        metrics["auc"] = ambient_predictions["auc"]
        metrics["loss"] = ambient_predictions["loss"]

        recall_at_cutoffs = (
            all_true_positives / (all_true_positives + all_false_negatives)
        )
        faph_at_cutoffs = ambient_false_positives / duration_of_ambient_set

        target_faph_cutoff_probability = 1.0
        recall_at_no_faph = 0
        for index, cutoff in enumerate(np.linspace(0.0, 1.0, 101)):
            if faph_at_cutoffs[index] == 0:
                target_faph_cutoff_probability = cutoff
                recall_at_no_faph = recall_at_cutoffs[index]
                break

        if faph_at_cutoffs[0] > 2:
            index_of_first_viable = 1
            while faph_at_cutoffs[index_of_first_viable] > 2:
                index_of_first_viable += 1
            x0 = faph_at_cutoffs[index_of_first_viable - 1]
            y0 = recall_at_cutoffs[index_of_first_viable - 1]
            x1 = faph_at_cutoffs[index_of_first_viable]
            y1 = recall_at_cutoffs[index_of_first_viable]
            recall_at_2faph = (y0 * (x1 - 2.0) + y1 * (2.0 - x0)) / (x1 - x0)
        else:
            index_of_first_viable = 0
            recall_at_2faph = recall_at_cutoffs[0]

        x_coordinates = [2.0]
        y_coordinates = [recall_at_2faph]
        for index in range(index_of_first_viable, len(recall_at_cutoffs)):
            if faph_at_cutoffs[index] != x_coordinates[-1]:
                x_coordinates.append(faph_at_cutoffs[index])
                y_coordinates.append(recall_at_cutoffs[index])

        average_viable_recall = (
            np.trapz(np.flip(y_coordinates), np.flip(x_coordinates)) / 2.0
        )

        metrics["recall_at_no_faph"] = recall_at_no_faph
        metrics["cutoff_for_no_faph"] = target_faph_cutoff_probability
        metrics["ambient_false_positives"] = ambient_false_positives[50]
        metrics["ambient_false_positives_per_hour"] = faph_at_cutoffs[50]
        metrics["average_viable_recall"] = average_viable_recall

    return metrics

_train_mod.validate_nonstreaming = _patched_validate
print('Keras 3 validate_nonstreaming patched in memory')

# Build the flags Namespace directly — same args as the CLI would parse
flags = argparse.Namespace(
    training_config='training_parameters.yaml',
    train=1,
    restore_checkpoint=0,
    test_tf_nonstreaming=0,
    test_tflite_nonstreaming=0,
    test_tflite_nonstreaming_quantized=0,
    test_tflite_streaming=0,
    test_tflite_streaming_quantized=1,
    use_weights='best_weights',
    verbosity=logging.INFO,
    model_name='mixednet',
    # mixednet architecture args
    pointwise_filters='64,64,64,64',
    repeat_in_block='1,1,1,1',
    mixconv_kernel_sizes='[5],[7,11],[9,15],[23]',
    residual_connection='0,0,0,0',
    first_conv_filters=32,
    first_conv_kernel_size=5,
    max_pool=0,
    spatial_attention=0,
    pooled=0,
    stride=3,
)

model_module = mixednet

print(f'Starting training (~{max(1, TRAINING_STEPS // 10000)} hour(s))...')
print('Steps print every 500 iterations.')
print('-' * 60)

try:
    # Load config from YAML
    config = mte.load_config(flags, model_module)
    data_processor = input_data.FeatureHandler(config)

    # Train
    if flags.train:
        model = model_module.model(
            flags, config['training_input_shape'], config['batch_size']
        )
        absl_logging.info(model.summary())
        mte.train_model(config, model, data_processor, flags.restore_checkpoint)

    # Evaluate + convert to TFLite
    if flags.test_tflite_streaming_quantized:
        model = model_module.model(
            flags, shape=config['training_input_shape'], batch_size=1
        )
        model.load_weights(
            os.path.join(config['train_dir'], flags.use_weights) + '.weights.h5'
        )
        absl_logging.info(model.summary())
        mte.evaluate_model(
            config, model, data_processor,
            flags.test_tf_nonstreaming,
            flags.test_tflite_nonstreaming,
            flags.test_tflite_nonstreaming_quantized,
            flags.test_tflite_streaming,
            flags.test_tflite_streaming_quantized,
        )

    print('-' * 60)
    print('Training complete!')
except Exception as e:
    print('-' * 60)
    print(f'Training raised exception: {type(e).__name__}: {e}')
    raise

## Step 13: Download Model

Downloads the trained `.tflite` model and ESPHome `.json` metadata.

If you configured `GITHUB_TOKEN` and `GITHUB_REPO` in Step 5, the model is also pushed to your repo.

In [ ]:
import os, glob, json, shutil

# ── Find the trained model ────────────────────────────────────
train_dir = 'trained_models/wakeword'
tflite_files = glob.glob(os.path.join(train_dir, '**/*.tflite'), recursive=True)

if not tflite_files:
    print('❌ No .tflite model found in trained_models/wakeword/')
    print('   Make sure Step 12 completed successfully.')
else:
    # Pick the quantized streaming model (preferred) or the first .tflite found
    quant_stream = [f for f in tflite_files if 'stream' in f and 'quant' in f]
    model_path = quant_stream[0] if quant_stream else tflite_files[0]
    print(f'Found model: {model_path}')
    model_size = os.path.getsize(model_path) / 1024
    print(f'Size: {model_size:.1f} KB')

    # ── Create ESPHome manifest JSON ──────────────────────────
    wake_word_clean = TARGET_WORD.replace('_', ' ').title()
    manifest = {
        "type": "micro",
        "model": os.path.basename(model_path),
        "author": "Custom (microWakeWord Colab)",
        "version": 1,
        "wake_word": wake_word_clean,
        "trained_languages": ["en"],
    }
    json_path = os.path.join(train_dir, f'{TARGET_WORD}.json')
    with open(json_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f'Manifest: {json_path}')

    # ── Copy to download directory ────────────────────────────
    dl_dir = 'download_model'
    os.makedirs(dl_dir, exist_ok=True)
    dl_model = os.path.join(dl_dir, f'{TARGET_WORD}.tflite')
    dl_json  = os.path.join(dl_dir, f'{TARGET_WORD}.json')
    shutil.copy2(model_path, dl_model)
    shutil.copy2(json_path, dl_json)

    # ── GitHub push (if configured) ───────────────────────────
    if GITHUB_TOKEN and GITHUB_REPO:
        import subprocess
        repo_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git'
        clone_dir = '/tmp/gh_push'
        if os.path.exists(clone_dir):
            shutil.rmtree(clone_dir)
        subprocess.run(['git', 'clone', '--depth', '1', repo_url, clone_dir],
                        check=True, capture_output=True)
        shutil.copy2(dl_model, clone_dir)
        shutil.copy2(dl_json, clone_dir)
        subprocess.run(['git', '-C', clone_dir, 'add', '.'], check=True)
        subprocess.run(['git', '-C', clone_dir, 'commit', '-m',
                        f'Add wake word model: {wake_word_clean}'],
                        check=True, capture_output=True)
        subprocess.run(['git', '-C', clone_dir, 'push'], check=True, capture_output=True)
        print(f'✅ Pushed to https://github.com/{GITHUB_REPO}')
    else:
        print('GitHub push: skipped (not configured)')

    # ── Download to your computer ─────────────────────────────
    try:
        from google.colab import files
        print(f'\nDownloading {TARGET_WORD}.tflite and {TARGET_WORD}.json...')
        files.download(dl_model)
        files.download(dl_json)
        print('✅ Check your browser downloads!')
    except ImportError:
        print(f'\nFiles ready in {dl_dir}/')
        print(f'  {dl_model}')
        print(f'  {dl_json}')